# 🎙️ TTS Text Optimizer for DesiVocal.com (Enhanced with Chunking)

**Optimize translated text with proper punctuation for natural voice generation.**

This notebook allows you to:
1.  **Setup Ollama**: Install and run Ollama locally in Colab.
2.  **Download Model**: Choose and pull a high-quality LLM (e.g., Qwen2.5, TranslateGemma).
3.  **Optimize Text**: Upload your text file, and the AI will add proper punctuation for natural TTS flow.
4.  **Download Result**: Save the optimized text as a `.txt` file for use on DesiVocal.com.

**✨ NEW: Supports large texts with automatic chunking and progress tracking!**

## 📦 Step 1: Install & Setup Ollama
Run this cell to install Ollama and start the server in the background.

In [ ]:
# Install required packages
!pip install -q ollama requests ipywidgets

# Install and start Ollama server
import subprocess
import time
import os
import sys

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running and ready!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")

## 📥 Step 2: Download Model
Select the model you want to use for optimization. `qwen2.5:14b` is recommended for high quality.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import ollama

print("🦙 Ollama Model Selection")
print("=" * 30)

# Model options
OLLAMA_MODELS = {
    "qwen2.5:14b (Recommended - High Quality)": "qwen2.5:14b",
    "qwen2.5:7b (Faster)": "qwen2.5:7b",
    "translategemma:27b (Very High Quality, Large)": "translategemma:27b",
    "mistral:7b (Good Quality)": "mistral:7b",
    "gemma2:9b (Google's Best Format)": "gemma2:9b"
}

model_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS.keys()),
    value="qwen2.5:14b (Recommended - High Quality)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(model_dropdown)
print("\nSelect a model and run this cell to download it.")

In [ ]:
# Pull the selected model
selected_model_name = OLLAMA_MODELS[model_dropdown.value]
print(f"📥 Pulling model: {selected_model_name}...")
print("   This may take a few minutes.")

try:
    # Pull with stream to show progress (simplified for non-interactive output)
    current_digest = ''
    for progress in ollama.pull(selected_model_name, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
             print() # Newline
        current_digest = digest
        
        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
             completed = progress['completed']
             total = progress['total']
             pct = (completed / total * 100) if total > 0 else 0
             print(f"\r   {status}: {pct:.1f}%", end='', flush=True)
        else:
             print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{selected_model_name}' ready to use!")
except Exception as e:
    print(f"\n❌ Error pulling model: {e}")

## 🧠 Step 3: Define Enhanced Optimizer Class with Chunking
This code defines the logic to communicate with Ollama and optimize the text with support for large texts.

In [ ]:
import requests
import json
import sys
import re
import time

class TTSOptimizer:
    """Optimizes text for desivocal.com TTS generation with chunking support"""
    
    def __init__(self, model_name="qwen2.5:14b", chunk_size=2000, timeout=600):
        """
        Initialize the TTS Optimizer
        
        Args:
            model_name: Name of the Ollama model to use
            chunk_size: Maximum characters per chunk (default: 2000)
            timeout: Request timeout in seconds (default: 600)
        """
        self.ollama_url = "http://localhost:11434/api/generate"
        self.model = model_name
        self.chunk_size = chunk_size
        self.timeout = timeout
        print(f"🤖 Initialized TTS Optimizer")
        print(f"   Model: {self.model}")
        print(f"   Chunk size: {self.chunk_size} chars")
        print(f"   Timeout: {self.timeout}s per chunk")

    def chunk_text(self, text: str) -> list:
        """
        Split text into chunks at sentence boundaries
        
        Args:
            text: Input text to chunk
            
        Returns:
            List of text chunks
        """
        if len(text) <= self.chunk_size:
            return [text]
        
        chunks = []
        current_chunk = ""
        
        # Split by common sentence endings (periods, question marks, exclamation marks)
        # This regex tries to split on sentence boundaries while preserving the punctuation
        sentences = re.split(r'([।॥.!?।]\s+)', text)
        
        for i in range(0, len(sentences), 2):
            sentence = sentences[i]
            separator = sentences[i+1] if i+1 < len(sentences) else ""
            
            # Check if adding this sentence would exceed chunk size
            if len(current_chunk) + len(sentence) + len(separator) > self.chunk_size and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence + separator
            else:
                current_chunk += sentence + separator
        
        # Add the last chunk if not empty
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        
        # If we still have no chunks (no sentence boundaries found), split by character count
        if not chunks:
            chunks = [text[i:i+self.chunk_size] for i in range(0, len(text), self.chunk_size)]
        
        print(f"\n📊 Split text into {len(chunks)} chunks")
        for idx, chunk in enumerate(chunks, 1):
            print(f"   Chunk {idx}: {len(chunk)} characters")
        
        return chunks

    def get_optimization_prompt(self, text: str, language: str = "Hindi") -> str:
        prompt = f"""You are a TTS punctuation expert optimizing text for natural voice generation on desivocal.com.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITICAL TASK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Add proper punctuation to {language} text for natural TTS voice flow.
This text will be used for voice generation - punctuation affects pauses and intonation.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHAT TO ADD:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. ✓ BASIC PUNCTUATION:
   - . (period) for statement endings
   - , (comma) for natural pauses
   - ? (question mark) for questions
   
2. ✓ EXPRESSIVE MARKS (use sparingly):
   - ??? for genuine questions/confusion
   - !!! for excitement/shock/strong emotion
   - ... for hesitation/suspense/trailing off

3. ✓ ABBREVIATIONS with dots:
   - AI → A.I., PhD → Ph.D., etc.

4. ✓ SENTENCE BREAKING (CRITICAL):
   - Break long sentences into shorter ones (10-20 words max)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT RULES (DO NOT VIOLATE):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✗ DO NOT change ANY words
✗ DO NOT add or remove content
✗ DO NOT translate anything
✗ DO NOT use SSML tags

✓ ONLY add punctuation marks: . , ? ! ??? !!! ...
✓ Keep 100% of original words intact

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INPUT TEXT:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{text}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT (TTS-OPTIMIZED VERSION):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return ONLY the optimized text with proper punctuation. No explanations."""
        return prompt
    
    def optimize_chunk(self, chunk: str, language: str = "Hindi", retry_count: int = 3) -> str:
        """
        Optimize a single chunk of text with retry logic
        
        Args:
            chunk: Text chunk to optimize
            language: Target language
            retry_count: Number of retries on failure
            
        Returns:
            Optimized text chunk
        """
        prompt = self.get_optimization_prompt(chunk, language)
        
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.4,
                "top_p": 0.9,
                "num_predict": -1
            }
        }
        
        for attempt in range(retry_count):
            try:
                response = requests.post(self.ollama_url, json=payload, timeout=self.timeout)
                response.raise_for_status()
                result = response.json()
                optimized_text = result.get("response", "").strip()
                # Clean output
                optimized_text = self._clean_output(optimized_text)
                return optimized_text
            except requests.exceptions.Timeout:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 10
                    print(f"\n⚠️ Timeout on attempt {attempt + 1}/{retry_count}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ Failed after {retry_count} attempts due to timeout")
                    raise
            except Exception as e:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 5
                    print(f"\n⚠️ Error on attempt {attempt + 1}/{retry_count}: {e}")
                    print(f"   Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ Failed after {retry_count} attempts: {e}")
                    raise
        
        return None
    
    def optimize(self, text: str, language: str = "Hindi") -> str:
        """
        Optimize text with automatic chunking for large texts
        
        Args:
            text: Input text to optimize
            language: Target language
            
        Returns:
            Optimized text
        """
        # Split text into chunks
        chunks = self.chunk_text(text)
        
        if len(chunks) == 1:
            print(f"\n📤 Processing single chunk ({len(text)} chars)...")
            return self.optimize_chunk(chunks[0], language)
        
        # Process multiple chunks
        print(f"\n🔄 Processing {len(chunks)} chunks...")
        optimized_chunks = []
        
        for idx, chunk in enumerate(chunks, 1):
            print(f"\n📤 Processing chunk {idx}/{len(chunks)} ({len(chunk)} chars)...")
            try:
                optimized = self.optimize_chunk(chunk, language)
                if optimized:
                    optimized_chunks.append(optimized)
                    print(f"✅ Chunk {idx}/{len(chunks)} complete!")
                else:
                    print(f"❌ Chunk {idx}/{len(chunks)} failed - using original")
                    optimized_chunks.append(chunk)
            except Exception as e:
                print(f"❌ Error processing chunk {idx}: {e}")
                print("   Using original chunk text")
                optimized_chunks.append(chunk)
        
        # Combine all chunks
        final_text = " ".join(optimized_chunks)
        print(f"\n✅ All chunks processed! Total output: {len(final_text)} characters")
        return final_text

    def _clean_output(self, text: str) -> str:
        """Clean the model output to remove formatting artifacts"""
        text = text.replace("```", "").replace("**", "")
        lines = [line.strip() for line in text.split('\n') 
                 if line.strip() and not line.strip().startswith('#') and not line.strip().startswith('OUTPUT')]
        return '\n'.join(lines).strip()

print("✅ TTSOptimizer class loaded with chunking support!")

## 📝 Step 4: Upload & Optimize
Upload your `.txt` file containing the text to optimize, select the language, and run the optimization.

In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

print("📂 Please upload your text file (.txt):")
uploaded = files.upload()

if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {uploaded_filename}")
else:
    print("⚠️ No file uploaded yet.")

language_input = widgets.Text(
    value='Hindi',
    placeholder='Target Language',
    description='Language:',
    layout=widgets.Layout(width='300px')
)

# Chunk size selector
chunk_size_input = widgets.IntSlider(
    value=2000,
    min=500,
    max=5000,
    step=500,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

print("\n⚙️ Configuration:")
display(language_input)
display(chunk_size_input)
print("\nℹ️ Chunk size: smaller = more API calls but less timeout risk")
print("   Recommended: 1500-2500 for large models, 2500-4000 for smaller models")

In [ ]:
# Run Optimization on Uploaded File
if not uploaded:
    print("⚠️ Please upload a file in the previous step first!")
else:
    # Read file content
    try:
        text_content = uploaded[uploaded_filename].decode("utf-8")
        print(f"📄 Read {len(text_content)} characters from file.")
        print(f"📏 File size: {len(text_content):,} characters")
        
        # Initialize optimizer with selected model and chunk size
        try:
            model_to_use = selected_model_name
        except NameError:
            model_to_use = "qwen2.5:14b" # Fallback
            print("⚠️ Using default model: qwen2.5:14b")
    
        optimizer = TTSOptimizer(
            model_name=model_to_use, 
            chunk_size=chunk_size_input.value,
            timeout=600  # 10 minutes per chunk
        )
        
        print(f"\n⏳ Starting optimization for {language_input.value}...")
        print("=" * 50)
        
        start_time = time.time()
        optimized_text = optimizer.optimize(text_content, language=language_input.value)
        end_time = time.time()
        
        processing_time = end_time - start_time
        
        if optimized_text:
            print("\n" + "=" * 50)
            print("✨ OPTIMIZATION COMPLETE!")
            print("=" * 50)
            print(f"⏱️  Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")
            print(f"📊 Input length: {len(text_content):,} chars")
            print(f"📊 Output length: {len(optimized_text):,} chars")
            print(f"📈 Size change: {((len(optimized_text) - len(text_content)) / len(text_content) * 100):+.1f}%")
            
            print("\n✨ Preview (First 800 characters):")
            print("=" * 50)
            print(optimized_text[:800])
            if len(optimized_text) > 800:
                print("\n... (truncated)")
            print("=" * 50)
            
            # Save to file
            output_filename = f"optimized_{uploaded_filename}"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(optimized_text)
                
            print(f"\n💾 Saved to: {output_filename}")
            print("📥 Downloading file...")
            
            # Trigger download
            files.download(output_filename)
            print("\n✅ Done! Check your downloads folder.")
            
        else:
            print("\n❌ Optimization failed. Please check the errors above.")
            
    except Exception as e:
        print(f"\n❌ Error reading or processing file: {e}")
        import traceback
        print("\n📋 Full error details:")
        print(traceback.format_exc())

## 🔧 Troubleshooting Tips

**If you're still getting timeouts:**

1. **Reduce chunk size**: Try 1000-1500 characters instead of 2000
2. **Use a smaller model**: Switch to `qwen2.5:7b` or `mistral:7b`
3. **Check Ollama server**: Run `!ollama ps` to see if the model is loaded
4. **Restart Ollama**: Go back to Step 1 and re-run the server setup

**For very large files (100k+ characters):**
- Consider splitting your file manually into smaller files first
- Use chunk size of 1000 characters or less
- The processing will take longer but will be more reliable